# **External Validation Dataset Analysis**
**Objectives:**

Verify rating definitions

Apply rating mapping

Analyze class distribution

Check overlap with existing movie reviews

Prepare external validation dataset summary

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive/crossDomainSentAnlys"

old_combined_dataset.csv    train_4domain_natural.csv
test_4domain_balanced.csv   train_with_neutral.csv
test_4domain_natural.csv    Turkish_movie_sentiment_dataset.csv
test_with_neutral.csv	    val_4domain_natural.csv
train_4domain_balanced.csv


In [3]:
import pandas as pd

external_df = pd.read_csv(
    "/content/drive/MyDrive/crossDomainSentAnlys/Turkish_movie_sentiment_dataset.csv"
)

In [4]:
print(external_df.shape)
print(external_df.columns.tolist())

external_df.head()

(83227, 3)
['comment', 'film_name', 'point']


,comment,film_name,point
0,\n Jean Reno denince zate...,Sevginin Gücü,"5,0"
1,\n Ekşın falan izlemek is...,Sevginin Gücü,"5,0"
2,\n Bu yapım hakkında öyle...,Sevginin Gücü,"5,0"
3,\n finali yeter... (sting...,Sevginin Gücü,"5,0"
4,\n Jean Reno..\r\nbu adam...,Sevginin Gücü,"5,0"


In [5]:
external_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83227 entries, 0 to 83226
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   comment    83227 non-null  object
 1   film_name  83227 non-null  object
 2   point      83227 non-null  object
dtypes: object(3)
memory usage: 1.9+ MB


In [6]:
external_df["point"].value_counts().sort_index()

,count
point,
"0,5",5150
"1,0",4917
"1,5",1464
"2,0",2433
"2,5",11433
"3,0",5499
"3,1",26
"3,2",38
"3,5",8565


In [7]:
"""negative_df = external_df[external_df["point"] <= 2]

positive_df = external_df[external_df["point"] >= 4]"""

'negative_df = external_df[external_df["point"] <= 2]\n\npositive_df = external_df[external_df["point"] >= 4]'

In [8]:
external_df["point"] = (
    external_df["point"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)


In [9]:
external_df["point"].dtype

dtype('float64')

In [10]:
negative_df = external_df[external_df["point"] <= 2]

positive_df = external_df[external_df["point"] >= 4]

filtered_df = pd.concat(
    [negative_df, positive_df],
    ignore_index=True
)


In [11]:
print("Original Reviews:", len(external_df))
print("Negative Reviews:", len(negative_df))
print("Positive Reviews:", len(positive_df))
print("Removed Reviews:", len(external_df) - len(filtered_df))
print("Final External Dataset Size:", len(filtered_df))

Original Reviews: 83227
Negative Reviews: 13964
Positive Reviews: 43336
Removed Reviews: 25927
Final External Dataset Size: 57300


In [12]:
import pandas as pd

train = pd.read_csv(
    "/content/drive/MyDrive/crossDomainSentAnlys/train_4domain_natural.csv"
)

val = pd.read_csv(
    "/content/drive/MyDrive/crossDomainSentAnlys/val_4domain_natural.csv"
)

test = pd.read_csv(
    "/content/drive/MyDrive/crossDomainSentAnlys/test_4domain_natural.csv"
)

In [13]:
movie_train = train[train["Domain"] == "Movie Reviews"]
movie_val = val[val["Domain"] == "Movie Reviews"]
movie_test = test[test["Domain"] == "Movie Reviews"]

In [14]:
our_movies = pd.concat(
    [movie_train, movie_val, movie_test],
    ignore_index=True
)

print(len(our_movies))

44804


In [15]:
our_movies["clean_text"] = (
    our_movies["text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [16]:
filtered_df["clean_text"] = (
    filtered_df["comment"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [17]:
our_set = set(our_movies["clean_text"])

external_set = set(filtered_df["clean_text"])


In [18]:
overlap = our_set.intersection(external_set)

print("Overlap Count:", len(overlap))


Overlap Count: 6894


In [19]:
list(overlap)[:20]

['güüüünaydın vietnam :d çok renkli çok güzel bir film izleyin derim.. :d sadece robin williamsın oyunculuğu için bile izlenir bu film..',
 'bence film çok basitti.. tamamen zaman kaybı..',
 'bu ne ya sibel can ve ali şan başta bu ikilinin beraber oynayabilecegi ne gibi bir film olabilir ki daha sonra bunların oyunculuklarını tartışalım.bence bu film sadece olmuş olsun diyer yapılıyo.',
 "dun aksam 3d olarak izledim ama 2d keyfi bile alamadim. 3d de yeni bir anlayis, gozlukle izliyorsun ama 2d hissediyorsun. filmin ilk 5 dakikasi 'gerilim budur' dedirtti fakat sonraki 100 dakikada 'galiba afisteki wes craven ismi yazim hatasi' diye dusundum. ikinci yari baslar baslamaz salondaki insanlar teker teker salondan cikmaya basladi, benim icin asil gerilim o zaman baslamisti, cunku salonda sadece ben ve arkadasim kalmistik. wes craven film isini beceremese bile gerilim isini biliyor, salonda 2 kisi kalinca gercek gerilimi yasiyorsun.kendine gel wes",
 'böyle saçma sapan bir film her zaman karş

In [20]:
print("Our movie reviews:", len(our_set))
print("External reviews:", len(external_set))
print("Overlap reviews:", len(overlap))

print(
    "Overlap ratio:",
    round(len(overlap) / len(external_set) * 100, 2),
    "%"
)

Our movie reviews: 44787
External reviews: 56731
Overlap reviews: 6894
Overlap ratio: 12.15 %


In [21]:
external_no_overlap = filtered_df[
    ~filtered_df["clean_text"].isin(overlap)
]

In [22]:
print("External dataset after overlap removal:",
      len(external_no_overlap))

External dataset after overlap removal: 50374


In [23]:
print("Original Reviews:", len(external_df))
print("Negative Reviews:", len(negative_df))
print("Positive Reviews:", len(positive_df))
print("Removed Reviews:", len(external_df) - len(filtered_df))
print("Filtered Reviews:", len(filtered_df))

Original Reviews: 83227
Negative Reviews: 13964
Positive Reviews: 43336
Removed Reviews: 25927
Filtered Reviews: 57300
